1. Chuẩn bị dữ liệu

In [ ]:
# Import thư viện và Đọc dữ liệu
import numpy as np

# B1. Đọc dữ liệu từ file CSV
data_bac_raw = np.genfromtxt('station_bac.csv', delimiter=',', dtype=str, skip_header=1)
data_nam_raw = np.genfromtxt('station_nam.csv', delimiter=',', dtype=str, skip_header=1)

print(f"Shape trạm Bắc: {data_bac_raw.shape}")
print(f"Shape trạm Nam: {data_nam_raw.shape}")

# B2. Tách cột và chuyển đổi kiểu dữ liệu (Nhiệm vụ 2 & 3) [cite: 12, 13]
# Cấu trúc: date, temp, humidity, rainfall

# Trạm Bắc
date_bac = np.array(data_bac_raw[:, 0], dtype='datetime64[D]')
temp_bac = data_bac_raw[:, 1].astype(float)
hum_bac  = data_bac_raw[:, 2].astype(float)
rain_bac = data_bac_raw[:, 3].astype(float)

# Trạm Nam
date_nam = np.array(data_nam_raw[:, 0], dtype='datetime64[D]')
temp_nam = data_nam_raw[:, 1].astype(float)
hum_nam  = data_nam_raw[:, 2].astype(float)
rain_nam = data_nam_raw[:, 3].astype(float)

# Kiểm tra nhanh 5 dòng đầu và 5 dòng cuối
print("\n--- Dữ liệu mẫu (Trạm Bắc) ---")
print("Date:", date_bac[:5])
print("Temp:", temp_bac[:5])

print("\n--- 5 dòng cuối (Trạm Bắc) ---")
print("Date:", date_bac[-5:])
print("Temp:", temp_bac[-5:])

Shape trạm Bắc: (180, 4)
Shape trạm Nam: (180, 4)

--- Dữ liệu mẫu (Trạm Bắc) ---
Date: ['2025-01-01' '2025-01-02' '2025-01-03' '2025-01-04' '2025-01-05']
Temp: [21.74 36.72 31.03 27.57 16.06]

--- 5 dòng cuối (Trạm Bắc) ---
Date: ['2025-06-25' '2025-06-26' '2025-06-27' '2025-06-28' '2025-06-29']
Temp: [16.53 29.96 22.06 36.35 15.58]


2. Thực hiện Mục tiêu 3: So sánh đặc điểm khí hậu giữa hai trạm

In [ ]:
# Mục tiêu 3: So sánh khí hậu giữa hai trạm theo thời gian.

print("=== MỤC TIÊU 3: SO SÁNH HAI TRẠM ===")

# 1. So sánh Nhiệt độ
# Sử dụng Broadcasting: Trừ trực tiếp hai mảng numpy
diff_temp = temp_bac - temp_nam 

# Tính toán các chỉ số so sánh
mean_diff_temp = np.mean(diff_temp)
days_bac_hotter = np.sum(diff_temp > 0) # Đếm số ngày Bắc nóng hơn Nam
days_nam_hotter = np.sum(diff_temp < 0) # Đếm số ngày Nam nóng hơn Bắc

print(f"\n1. Chênh lệch Nhiệt độ trung bình (Bắc - Nam): {mean_diff_temp:.2f} độ C")
print(f"- Số ngày Trạm Bắc nóng hơn: {days_bac_hotter} ngày")
print(f"- Số ngày Trạm Nam nóng hơn: {days_nam_hotter} ngày")

# 2. So sánh Lượng mưa
total_rain_bac = np.sum(rain_bac)
total_rain_nam = np.sum(rain_nam)
diff_rain = total_rain_bac - total_rain_nam

print(f"\n2. Tổng lượng mưa 6 tháng:")
print(f"- Trạm Bắc: {total_rain_bac:.2f} mm")
print(f"- Trạm Nam: {total_rain_nam:.2f} mm")
print(f"- Kết luận: Trạm {'Bắc' if diff_rain > 0 else 'Nam'} mưa nhiều hơn {abs(diff_rain):.2f} mm")

# 3. So sánh Độ ẩm
# Dùng Boolean mask để tìm những ngày cả hai trạm đều có độ ẩm cao (>80%)
high_hum_both = (hum_bac > 80) & (hum_nam > 80)
count_high_hum = np.sum(high_hum_both)

print(f"\n3. Độ ẩm đồng thời:")
print(f"- Số ngày cả hai trạm cùng có độ ẩm > 80%: {count_high_hum} ngày")

=== MỤC TIÊU 3: SO SÁNH HAI TRẠM ===

1. Chênh lệch Nhiệt độ trung bình (Bắc - Nam): -3.71 độ C
- Số ngày Trạm Bắc nóng hơn: 69 ngày
- Số ngày Trạm Nam nóng hơn: 111 ngày

2. Tổng lượng mưa 6 tháng:
- Trạm Bắc: 17486.00 mm
- Trạm Nam: 19845.40 mm
- Kết luận: Trạm Nam mưa nhiều hơn 2359.40 mm

3. Độ ẩm đồng thời:
- Số ngày cả hai trạm cùng có độ ẩm > 80%: 19 ngày


3. Thực hiện Mục tiêu 4: Nhận diện giai đoạn biến động mạnh

In [ ]:
# Mục tiêu 4: Xác định những giai đoạn biến động mạnh của nhiệt độ/độ ẩm/lượng mưa.

print("\n=== MỤC TIÊU 4: NHẬN DIỆN BIẾN ĐỘNG MẠNH ===")

def phan_tich_bien_dong(data_array, date_array, threshold, column_name, station_name):
    """
    Hàm tính toán biến động ngày-qua-ngày sử dụng np.diff
    """
    # np.diff tính hiệu số: ngày (i+1) - ngày i
    # Kết quả sẽ ít hơn mảng gốc 1 phần tử
    changes = np.diff(data_array)
    
    # Lấy giá trị tuyệt đối của biến động
    abs_changes = np.abs(changes)
    
    # Tạo Boolean mask dựa trên ngưỡng
    mask_fluctuation = abs_changes > threshold
    
    # Lấy các ngày xảy ra biến động
    dates_fluctuation = date_array[1:][mask_fluctuation]
    values_fluctuation = changes[mask_fluctuation]
    
    print(f"\n[Trạm {station_name}] Biến động {column_name} (Ngưỡng > {threshold}):")
    print(f"- Tổng số ngày biến động mạnh: {len(dates_fluctuation)}")
    
    if len(dates_fluctuation) > 0:
        # In tối đa 5 ngày đầu tiên làm mẫu
        print("- Chi tiết 5 ngày đầu:")
        for date, val in zip(dates_fluctuation[:5], values_fluctuation[:5]):
            status = "Tăng" if val > 0 else "Giảm"
            print(f"  + {date}: {status} {abs(val):.2f}")

# --- Áp dụng cho Trạm Bắc ---
phan_tich_bien_dong(temp_bac, date_bac, 8, "Nhiệt độ", "Bắc")
phan_tich_bien_dong(hum_bac, date_bac, 15, "Độ ẩm", "Bắc")
phan_tich_bien_dong(rain_bac, date_bac, 40, "Lượng mưa", "Bắc")

# --- Áp dụng cho Trạm Nam ---
print("-" * 30)
phan_tich_bien_dong(temp_nam, date_nam, 8, "Nhiệt độ", "Nam")
phan_tich_bien_dong(hum_nam, date_nam, 15, "Độ ẩm", "Nam")
phan_tich_bien_dong(rain_nam, date_nam, 40, "Lượng mưa", "Nam")


=== PHÂN TÍCH MỤC TIÊU 4: NHẬN DIỆN BIẾN ĐỘNG MẠNH ===

[Trạm Bắc] Biến động Nhiệt độ (Ngưỡng > 8):
- Tổng số ngày biến động mạnh: 84
- Chi tiết 5 ngày đầu:
  + 2025-01-02: Tăng 14.98
  + 2025-01-05: Giảm 11.51
  + 2025-01-08: Tăng 21.01
  + 2025-01-11: Giảm 17.87
  + 2025-01-12: Tăng 24.68

[Trạm Bắc] Biến động Độ ẩm (Ngưỡng > 15):
- Tổng số ngày biến động mạnh: 87
- Chi tiết 5 ngày đầu:
  + 2025-01-03: Tăng 44.70
  + 2025-01-05: Giảm 34.10
  + 2025-01-06: Tăng 22.10
  + 2025-01-10: Giảm 15.80
  + 2025-01-12: Tăng 44.20

[Trạm Bắc] Biến động Lượng mưa (Ngưỡng > 40):
- Tổng số ngày biến động mạnh: 107
- Chi tiết 5 ngày đầu:
  + 2025-01-02: Tăng 45.90
  + 2025-01-05: Tăng 71.30
  + 2025-01-06: Giảm 100.00
  + 2025-01-07: Tăng 103.50
  + 2025-01-09: Giảm 127.80
------------------------------

[Trạm Nam] Biến động Nhiệt độ (Ngưỡng > 8):
- Tổng số ngày biến động mạnh: 59
- Chi tiết 5 ngày đầu:
  + 2025-01-03: Giảm 8.31
  + 2025-01-04: Tăng 9.75
  + 2025-01-08: Tăng 12.27
  + 2025-01-16: G